## Importing the Necessary Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

# Using seaborn set_style function for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## Loading the Data

In [2]:
data = pd.read_csv('kmeans_data/data.csv', header=None)
labels = pd.read_csv('kmeans_data/label.csv', header=None)

In [3]:
data

,0,1,2,3,4,5,6,7,8,9,...,774,775,776,777,778,779,780,781,782,783
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
labels

,0
0,7
1,2
2,1
3,0
4,4
...,...
9995,2
9996,3
9997,4
9998,5


In [5]:
print(data.shape)
print(labels.shape)

(10000, 784)
(10000, 1)


## Converting the Data and Labels to NumPy Arrays

In [6]:
X = data.values
y = labels.values.flatten()

In [7]:
print(X.dtype)
print(y.dtype)

int64
int64


## Data Exploration

In [8]:
print("Number of samples: ", X.shape[0])
print("Number of features: ", X.shape[1])


Number of samples:  10000
Number of features:  784


In [9]:
print("Unique Classes in Labels: ", np.unique(y))
print("Number of Unique Classes: ", len(np.unique(y)))

Unique Classes in Labels:  [0 1 2 3 4 5 6 7 8 9]
Number of Unique Classes:  10


In [10]:
unique_classes, counts = np.unique(y, return_counts=True)

In [11]:
for label, count in zip(unique_classes, counts):
    perecentage = (count / len(y)) * 100 
    print(f"Class {label}: {count} samples, {perecentage:.2f}% of total")

K = len(unique_classes)
print(f"Total number of clusters (K): {K}")


Class 0: 980 samples, 9.80% of total
Class 1: 1135 samples, 11.35% of total
Class 2: 1032 samples, 10.32% of total
Class 3: 1010 samples, 10.10% of total
Class 4: 982 samples, 9.82% of total
Class 5: 892 samples, 8.92% of total
Class 6: 958 samples, 9.58% of total
Class 7: 1028 samples, 10.28% of total
Class 8: 974 samples, 9.74% of total
Class 9: 1009 samples, 10.09% of total
Total number of clusters (K): 10


## Building the K-Means Algorithm from scratch

We are required to build the K-Means algorithm using three different distance metrices: Euclidean, Cosine Similarity and Generalized Jaccard similarity. 

In [12]:
# First we will implement the Euclidean distance function

def euclidean_distance(point1, point2):
    """Calculate the Euclidean distance between two points."""
    return np.sqrt(np.sum((point1 - point2) ** 2))

In [13]:
# Now we will build the K-Means algorithm using Cosine Similarity metric

def cosine_distance(point1, point2):
    """Calculate the Cosine Similarity between two points."""

    # Calculating the dot product between the two points.
    dot_product = np.dot(point1, point2)

    norm1 = np.linalg.norm(point1)
    norm2 = np.linalg.norm(point2)

    # Avoid division by zero
    if norm1 == 0 or norm2 == 0:
        return 1.0
    
    cosine_similarity = dot_product / (norm1 * norm2)

    return 1 - cosine_similarity


In [14]:
# Implementing Generalized Jaccard Similarity

def jaccard_distance(point1, point2):
    """ Calculate the Generalized Jaccard Similarity between two points."""
    numerator = np.sum(np.minimum(point1, point2))
    denominator = np.sum(np.maximum(point1, point2))

    # Avoid division by zero
    if denominator == 0:
        return 1.0
    
    jaccard_similarity = numerator / denominator
    return 1 - jaccard_similarity


## Randomly Initializing Centroids for K-Means Algorithm

In [15]:
def initialize_centroids(X, k, random_seed=42):
    """ Randomly initialize K centroids fron the dataset X."""
    
    np.random.seed(random_seed)

    random_indices = np.random.choice(X.shape[0], size=k, replace=False)

    centroids = X[random_indices].copy()

    return centroids

In [16]:
# Assigning the Points to Nearest Centroid

def assign_clusters(X, centroids, distance_function):
    """ Assign the data points to the nearest centroids based on the given distance function. """

    n_samples = X.shape[0]
    labels = np.zeros(n_samples, dtype=int)

    for i in range(n_samples):
        # Calculate the distance from the point to each of the centroids
        distances = [distance_function(X[i], centroid) for centroid in centroids]

        # Assigning the points to the nearest centroids
        labels[i] = np.argmin(distances)
    
    return labels


## Updating the Centroids based on the Mean

In [17]:
def update_centroids(X, labels, k):
    """ Update the centroids for each of the clusters based on the mean of the newly assigned points. """

    n_features = X.shape[1]
    new_centroids = np.zeros((k, n_features))

    for i in range(k):
        # Getting all the points in that cluster
        cluster_points = X[labels == i]

        if len(cluster_points) > 0:
            # Computing the mean of the cluster points to update the centroid value
            new_centroids[i] = cluster_points.mean(axis=0)
        else:
            # If a cluster has no points assigned, reinitialize its centroid randomly
            new_centroids[i] = cluster_points.mean(axis=0)
    
    return new_centroids


## Calculating the Sum of Squared Errors for all the Distance Metrices

In [18]:
def calculate_sse(X, labels, centroids, distance_function):
    """ Calculate the Sum of Squared Errors for all the distance metrices. """
    
    sse = 0.0

    for i in range(len(centroids)):
        # Getting the cluster points
        mask = (labels == i)
        cluster_points = X[mask]

        # Calculating the squared distance from each point to the centroid
        for point in cluster_points:
            sse += distance_function(point, centroids[i]) ** 2  

    return sse

In [19]:
# Define the k for K-Means Algorithm
K = len(np.unique(y))

## Compiling the K-Means Algorithm

In [20]:
def kmeans(X, k, distance_function, max_iterations = 500, random_seed=42):
    """ K-Means Clustering Algorithm Implementation. """

    centroids = initialize_centroids(X, k, random_seed)

    sse_history = []
    prev_sse = float('inf')

    for iteration in range(max_iterations):

        # Assigning clusters based on the nearest centroids
        labels = assign_clusters(X, centroids, distance_function)

        # Calculating the Current Sum of Squared Errors
        current_sse = calculate_sse(X, labels, centroids, distance_function)
        sse_history.append(current_sse)

        # Checking if the current SSE is more than previous SSS
        if current_sse > prev_sse:
            print(f"Stopped as there was increase in SSE at iteration {iteration}")
            break

        new_centroids = update_centroids(X, labels, k)

        # Checking if the current SSE is equal to the previous SSE
        if np.allclose(centroids, new_centroids):
            print(f"Converged at iteration {iteration}")
            iteration += 1
            break

            centroids = new_centroids
            prev_sse = current_sse

    else:
        print(f"Stopped after reaching maximum iterations: {max_iterations}")
        iteration = max_iterations

    return labels, centroids, current_sse, iteration, sse_history


## Running the K-Means Algorithm with all the 3 distance metrices

In [21]:
# 1. Euclidean Distance Metric

K = len(np.unique(y))

start_time = time.time()
euclidean_labels, euclidean_centroids, euclidean_sse, euclidean_iter, euclidean_sse_history = kmeans(X, K, euclidean_distance, max_iterations=500, random_seed=42)
euclidean_time = time.time() - start_time

print("Euclidean Distance Metric Results:")
print(f" Final SSE: {euclidean_sse}")
print(f" Number of iterations: {euclidean_iter}")
print(f" Time taken: {euclidean_time:.4f} seconds")

Stopped after reaching maximum iterations: 500
Euclidean Distance Metric Results:
 Final SSE: 46477490030.0
 Number of iterations: 500
 Time taken: 206.5972 seconds


In [22]:
# 2. Cosine Distance Metric

start_time = time.time()
cosine_labels, cosine_centroids, cosine_sse, cosine_iter, cosine_sse_history = kmeans(X, K, cosine_distance, max_iterations=500, random_seed=42)
cosine_time = time.time() - start_time

print("Cosine Similarity Distance Metric Results:")
print(f" Final SSE: {cosine_sse}")
print(f" Number of iterations: {cosine_iter}")
print(f" Time taken: {cosine_time:.4f} seconds")

Stopped after reaching maximum iterations: 500
Cosine Similarity Distance Metric Results:
 Final SSE: 1794.3985619888995
 Number of iterations: 500
 Time taken: 288.1782 seconds


In [23]:
# 3. Generalized Jaccard Distance Metric
start_time = time.time()
jaccard_labels, jaccard_centroids, jaccard_sse, jaccard_iter, jaccard_sse_history = kmeans(X, K, jaccard_distance, max_iterations=500, random_seed=42)
jaccard_time = time.time() - start_time

print("Generalized Jaccard Distance Metric Results:")
print(f" Final SSE: {jaccard_sse}")
print(f" Number of iterations: {jaccard_iter}")
print(f" Time taken: {jaccard_time:.4f} seconds")

Stopped after reaching maximum iterations: 500
Generalized Jaccard Distance Metric Results:
 Final SSE: 4196.271346153514
 Number of iterations: 500
 Time taken: 257.3101 seconds
